In [28]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv('MONGODB_URI'))

# Lister toutes les bases
db_list = client.list_database_names()

for db_name in db_list:
    print(f"\n📌 Base : {db_name}")
    db = client[db_name]

    # Lister les collections de cette base
    collections = db.list_collection_names()

    if not collections:
        print("  (aucune collection)")
        continue

    for col in collections:
        count = db[col].count_documents({})
        print(f"  - {col} : {count} documents")

db = client['flight_delay_history_db']

#cursor = db['aviationstack_historical_landed_flights'].find().limit(100)
#df_aviation = pd.DataFrame(list(cursor))
#df_aviation.head()




📌 Base : flight_delay_db
  - aviationstack_flights : 1543 documents
  - afklm_flights : 1000 documents
  - weather_data : 78 documents

📌 Base : flight_delay_history_db
  - aviationstack_historical_landed_flights : 27 documents

📌 Base : flight_delays_test
  - test_collection : 22 documents

📌 Base : admin
  (aucune collection)

📌 Base : local
  - oplog.rs : 286 documents


In [ ]:
# Copier les bons vols
# A UTILISER UNE FOIS
# Source et destination
source_db = client['flight_delay_db']
target_db = client['flight_delay_history_db']

source_collection = source_db['aviationstack_flights']
target_collection = target_db['aviationstack_historical_landed_flights']

# Filtre : vols atterris avec arrival.actual non null
query = {
    "flight_status": "landed",
    "arrival.actual": {"$nin": [None, ""]}
}

# Récupération des documents filtrés
filtered_docs = list(source_collection.find(query))

# Insertion dans la collection cible
if filtered_docs:
    result = target_collection.insert_many(filtered_docs)
    print(f"{len(result.inserted_ids)} documents copiés.")
else:
    print("Aucun document à copier.")


24 documents copiés.


In [2]:
from pprint import pprint

db = client['flight_delay_history_db']

collection = db['aviationstack_historical_landed_flights']

# condition
query = {
    "flight_status": "landed",
    "arrival.actual": {"$ne": None}
}

# Compter les documents
count = collection.count_documents(query)
print("Nombre de documents filtrés :", count)

# Afficher les 1 derniers
for doc in collection.find(query).sort("collected_at", -1).limit(1):
    pprint(doc)


Nombre de documents filtrés : 26
{'_id': 'U24835_2026-02-01',
 'aircraft': {'iata': None,
              'icao': None,
              'icao24': '440037',
              'registration': None},
 'airline': {'iata': 'U2', 'icao': 'EZY', 'name': 'easyJet'},
 'arrival': {'actual': '2026-02-01T15:46:00+00:00',
             'actual_runway': '2026-02-01T15:46:00+00:00',
             'airport': 'Linate',
             'baggage': None,
             'delay': None,
             'estimated': '2026-02-01T15:54:00+00:00',
             'estimated_runway': '2026-02-01T15:46:00+00:00',
             'gate': None,
             'iata': 'LIN',
             'icao': 'LIML',
             'scheduled': '2026-02-01T15:50:00+00:00',
             'terminal': None,
             'timezone': 'Europe/Rome'},
 'collected_at': datetime.datetime(2026, 2, 1, 16, 38, 3, 895000),
 'departure': {'actual': '2026-02-01T14:40:00+00:00',
               'actual_runway': '2026-02-01T14:40:00+00:00',
               'airport': 'Orly',
  

In [4]:
from pymongo import MongoClient
import os
from dotenv import load_dotenv

load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))
db = client["flight_delay_db"]
collection = db["aviationstack_flights"]

query = {
    "arrival.actual": {"$nin": [None, ""]}
}

count = collection.count_documents(query)
print("Nombre de documents :", count)


Nombre de documents : 24


In [29]:
# test ML pas à pas
import os
from dotenv import load_dotenv
from pymongo import MongoClient
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import f1_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import matplotlib.pyplot as plt
%matplotlib inline

# ETAPE 0  : RECUPERATION DES DONNEES
load_dotenv()

client = MongoClient(os.getenv("MONGODB_URI"))
db = client["flight_delay_history_db"]
collection = db["aviationstack_historical_landed_flights"]

# Filtre : vols atterris avec données complètes minimales
query = {
    "flight_status": "landed",
    "arrival.actual": {"$nin": [None, ""]},
    "arrival.scheduled": {"$nin": [None, ""]},
    "departure.scheduled": {"$nin": [None, ""]},
    "departure.actual": {"$nin": [None, ""]},
}

docs = list(collection.find(query))



# ETAPE 1  : APPLATISSEMENT DES DONNEES
# 🔹 Aplatir les documents imbriqués
df_flat = pd.json_normalize(
    docs,
    sep="_"  # airline.name -> airline_name
)

# 🔹 Garder / renommer les colonnes utiles
cols_to_keep = [
    "_id",
    "flight_date",
    "flight_status",
    "collected_at",
    "filtered_at",
    "live",

    # Airline
    "airline_name",
    "airline_iata",
    "airline_icao",

    # Aircraft
    "aircraft_registration",
    "aircraft_iata",
    "aircraft_icao",
    "aircraft_icao24",

    # Departure
    "departure_airport",
    "departure_timezone",
    "departure_iata",
    "departure_icao",
    "departure_terminal",
    "departure_gate",
    "departure_scheduled",
    "departure_estimated",
    "departure_actual",
    "departure_estimated_runway",
    "departure_actual_runway",

    # Arrival
    "arrival_airport",
    "arrival_timezone",
    "arrival_iata",
    "arrival_icao",
    "arrival_terminal",
    "arrival_gate",
    "arrival_baggage",
    "arrival_scheduled",
    "arrival_estimated",
    "arrival_actual",
    "arrival_estimated_runway",
    "arrival_actual_runway",

    # Flight
    "flight_number",
    "flight_iata",
    "flight_icao",
    "flight_codeshared",
]

# Certaines colonnes peuvent ne pas exister pour tous les docs → on intersecte
existing_cols = [c for c in cols_to_keep if c in df_flat.columns]
df_flat = df_flat[existing_cols]

# Optionnel : conversion de quelques champs en datetime
datetime_cols = [
    "flight_date",
    "collected_at",
    "filtered_at",
    "departure_scheduled",
    "departure_estimated",
    "departure_actual",
    "departure_estimated_runway",
    "departure_actual_runway",
    "arrival_scheduled",
    "arrival_estimated",
    "arrival_actual",
    "arrival_estimated_runway",
    "arrival_actual_runway",
]

for col in datetime_cols:
    if col in df_flat.columns:
        df_flat[col] = pd.to_datetime(df_flat[col], errors="coerce")

# df_flat.head()

# ETAPE 2  : ANALYSE DES VALEURS NULLS ET SUPPRESSIONS DES COLONNES INUTILES
# Nombre de valeurs nulles par colonne
null_counts = df_flat.isnull().sum()

# Pourcentage de valeurs nulles par colonne
null_percent = (null_counts / len(df_flat)) * 100

# Résumé filtré : uniquement les colonnes avec plus de 30% de valeurs nulles
null_summary = (
    pd.DataFrame({
        "null_count": null_counts,
        "null_percent": null_percent.round(2)
    })
    .query("null_percent > 0")
    .sort_values(by="null_percent", ascending=False)
)

display(null_summary)

# On supprimme les colonnes qui ont plus de 30% de valeurs nulls
# Colonnes à supprimer : null_percent > 30%
cols_to_drop = null_summary[null_summary["null_percent"] > 30.0].index.tolist()

# Nouveau DataFrame nettoyé
df_flat_filtered = df_flat.drop(columns=cols_to_drop)

print(f"Colonnes supprimées : {len(cols_to_drop)}")
print(cols_to_drop)

# Suppressions des colonnes runway car inutiles
cols_runway = [col for col in df_flat_filtered.columns if "runway" in col.lower()]

df_flat_filtered = df_flat_filtered.drop(columns=cols_runway)

print("Colonnes runway supprimées :", cols_runway)

# df_flat_filtered.head()

# ETAPE 3  : TRAITEMENT DES VALEURS NULLS
# Ici c'est la colonne departure_terminal
display(df_flat_filtered['departure_terminal'].value_counts()) 
mode_value = df_flat_filtered['departure_terminal'].mode()[0]
df_flat_filtered['departure_terminal'] = df_flat_filtered['departure_terminal'].fillna(mode_value)

# Vérification si il y a encore des valeurs nulls
df_flat_filtered.isnull().any().any()

# ETAPE 4 : CALCUL DES DÉLAIS
# Fonction utilitaire pour calculer un délai en minutes
def compute_delay(actual, scheduled):
    if pd.isna(actual) or pd.isna(scheduled):
        return None
    delay = (actual - scheduled).total_seconds() / 60
    return max(delay, 0)  # clamp à 0

# Délais départ
df_flat_filtered["departure_delay_actual"] = df_flat_filtered.apply(
    lambda row: compute_delay(row["departure_actual"], row["departure_scheduled"]),
    axis=1
)

df_flat_filtered["departure_delay_estimated"] = df_flat_filtered.apply(
    lambda row: compute_delay(row["departure_estimated"], row["departure_scheduled"]),
    axis=1
)

# Délais arrivée
df_flat_filtered["arrival_delay_actual"] = df_flat_filtered.apply(
    lambda row: compute_delay(row["arrival_actual"], row["arrival_scheduled"]),
    axis=1
)

df_flat_filtered["arrival_delay_estimated"] = df_flat_filtered.apply(
    lambda row: compute_delay(row["arrival_estimated"], row["arrival_scheduled"]),
    axis=1
)

# Vérification rapide
df_flat_filtered[[
    "departure_delay_actual",
    "departure_delay_estimated",
    "arrival_delay_actual",
    "arrival_delay_estimated"
]].sort_values(by="arrival_delay_actual", ascending=False).head()

df_flat_filtered.sort_values(by="arrival_delay_actual", ascending=False).head()


,null_count,null_percent
live,27,100.00
aircraft_registration,27,100.00
aircraft_iata,27,100.00
flight_codeshared,27,100.00
aircraft_icao,27,100.00
arrival_gate,27,100.00
filtered_at,25,92.59
departure_gate,25,92.59
arrival_baggage,18,66.67
arrival_terminal,9,33.33


Colonnes supprimées : 10
['live', 'aircraft_registration', 'aircraft_iata', 'flight_codeshared', 'aircraft_icao', 'arrival_gate', 'filtered_at', 'departure_gate', 'arrival_baggage', 'arrival_terminal']
Colonnes runway supprimées : ['departure_estimated_runway', 'departure_actual_runway', 'arrival_estimated_runway', 'arrival_actual_runway']


departure_terminal
1    20
3     3
4     1
Name: count, dtype: int64

,_id,flight_date,flight_status,collected_at,airline_name,airline_iata,airline_icao,aircraft_icao24,departure_airport,departure_timezone,...,arrival_scheduled,arrival_estimated,arrival_actual,flight_number,flight_iata,flight_icao,departure_delay_actual,departure_delay_estimated,arrival_delay_actual,arrival_delay_estimated
23,U24184_2025-12-22,2025-12-22,landed,2025-12-22 22:54:03.316,easyJet,U2,EZY,4403BB,Orly,Europe/Paris,...,2025-12-22 22:35:00+00:00,2025-12-22 22:56:00+00:00,2025-12-22 22:56:00+00:00,4184,U24184,EZY4184,39.0,0.0,21.0,21.0
26,U29043_2026-02-03,2026-02-03,landed,2026-02-03 18:01:52.007,easyJet,U2,EZY,440CCC,Orly,Europe/Paris,...,2026-02-03 15:55:00+00:00,2026-02-03 16:13:00+00:00,2026-02-03 16:15:00+00:00,9043,U29043,EZY9043,54.0,0.0,20.0,18.0
12,AF4466_2025-12-22,2025-12-22,landed,2025-12-22 22:54:02.759,Air France,AF,AFR,39DD42,Orly,Europe/Paris,...,2025-12-22 22:40:00+00:00,2025-12-22 22:43:00+00:00,2025-12-22 22:49:00+00:00,4466,AF4466,AFR4466,9.0,0.0,9.0,3.0
19,XK787_2025-12-22,2025-12-22,landed,2025-12-22 22:54:02.994,Air Corsica,XK,CCM,39DD42,Orly,Europe/Paris,...,2025-12-22 22:40:00+00:00,2025-12-22 22:43:00+00:00,2025-12-22 22:49:00+00:00,787,XK787,CCM787,9.0,0.0,9.0,3.0
24,U24092_2025-12-22,2025-12-22,landed,2025-12-22 22:54:03.337,easyJet,U2,EZY,44061D,Orly,Europe/Paris,...,2025-12-22 22:25:00+00:00,2025-12-22 22:26:00+00:00,2025-12-22 22:29:00+00:00,4092,U24092,EZY4092,21.0,0.0,4.0,1.0
